# Vector Search によるセマンティック検索

**学習目標**
  1. Embeddings API を使用してテキスト埋め込みを作成する方法を学ぶ
  1. Vector Search に埋め込みをロードする方法を学ぶ
  2. Vector Search をクエリする方法を学ぶ
  1. セマンティック マッチに基づいて情報検索システムを構築する方法を学ぶ
  
  
このノートブックでは、[Vector Search](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/overview) と [Text Embeddings API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/reference/models/text-embeddings-api) を使用して、シンプルでありながら高速かつスケーラブルな[セマンティック検索](https://en.wikipedia.org/wiki/Semantic_search#:~:text=Semantic%20search%20seeks%20to%20improve,to%20generate%20more%20relevant%20results.)検索システムを実装します。セマンティック検索システムでは、ユーザーのクエリに対して、セマンティックな一致度順にランク付けされた複数のドキュメントが返されます。これは、ブール検索やキーワードベースの検索システムとは異なり、返されるドキュメントがクエリの実際の正確なキーワードではなく、意図や意味に一致する必要があることを意味します。このようなセマンティック検索システムには、一般に次の 2 つのコンポーネントがあります。

* ドキュメントとユーザー クエリの両方のセマンティックに意味のあるベクトル表現を生成するコンポーネント。大規模言語モデルの力を活用してこれらの埋め込みを作成するために、[Text Embeddings API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/reference/models/text-embeddings-api) を使用します。

* ユーザーがドキュメントのベクトル埋め込みを保存し、埋め込み空間内でユーザー クエリの埋め込みに最も近い埋め込みを持つドキュメントを返すことで、最も関連性の高いドキュメントを取得できるようにするコンポーネント。[Google Research の最近の論文](https://arxiv.org/abs/1908.10396) に基づいて、クエリ ベクトルに最も近いドキュメント ベクトルを比較して取得するための[効率的な近似最近傍戦略](https://ai.googleblog.com/2020/07/announcing-scann-efficient-vector.html) により、数十億の埋め込みまでスケールアップできる [Vector Search](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/overview) を使用します。



**データセット:** COVID-19 に焦点を当てた約 100 万件の医学研究論文を含む [COVID-19 Open Research Dataset Challenge (CORD-19)](https://www.kaggle.com/datasets/allen-institute-for-ai/CORD-19-research-challenge) の非常に小さなサブセットを使用します。処理速度を考慮して、2021 年の 4,000 件のタイトル、要約（abstract）、URL のみに焦点を当てます。

## セットアップ

In [ ]:
import json
import os

import pandas as pd
from google import genai
from google.cloud import aiplatform
from IPython import display

In [ ]:
REGION = "us-central1"
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = f"{PROJECT}-cord19-semantic-search"

# これらを変更しないでください
os.environ["PROJECT"] = PROJECT
os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION


In [ ]:
!gcloud storage ls --buckets | grep -w gs://{BUCKET}/ || gcloud storage buckets create --location={REGION} gs://{BUCKET}

## データのロード

使用するデータセットは、[COVID-19 Open Research Dataset Challenge (CORD-19)](https://www.kaggle.com/datasets/allen-institute-for-ai/CORD-19-research-challenge) に含まれる約 100 万件の医学論文から抽出した約 4,000 件のサンプルのタイトル、要約、URL メタデータです。このラボでは、要約をドキュメントとして使用し、埋め込みを計算して保存します。

In [ ]:
metadata = pd.read_csv("../data/cord19_metadata_sample.csv.gz")
metadata = metadata[~metadata.abstract.isna()]
metadata.index = range(len(metadata))
metadata.head()

## 埋め込みの作成

最初に行うことは、要約の埋め込みベクトルを作成することです。そのためには、まず Gen AI SDK クライアントをインスタンス化する必要があります。

In [ ]:
client = genai.Client(vertexai=True, location="us-central1")
EMBEDDING_MODEL = "text-embedding-004"

この埋め込みモデルは、一度に最大 5 つのテキストのリストを処理できます。そのため、`metadata.abstract` を 5 件ずつのバッチで反復処理し、これらのバッチを `client.models.embed_content` に渡してすべての要約の埋め込みを作成し、リスト `vectors` に保存します。次のセルの実行には数分かかります。

In [ ]:
MAX_BATCH_SIZE = 5
vectors = []

for i in range(0, len(metadata), MAX_BATCH_SIZE):
    batch = metadata.abstract[i : i + MAX_BATCH_SIZE].to_list()
    embeddings = client.models.embed_content(
        model=EMBEDDING_MODEL, contents=batch
    )
    vectors.extend([embedding.values for embedding in embeddings.embeddings])

## Vector Search エンジンの入力ファイルの作成

この時点で、要約の埋め込みはメモリ内の `vectors` リストに保存されています。これらの埋め込みを [Vector Search](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/overview) に保存するには、[次の形式](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/format-structure) の JSON ファイルにシリアル化する必要があります。

```python
{"id": <DOCUMENT_ID1>, "embedding": [0.1, ..., -0.7]}
{"id": <DOCUMENT_ID2>, "embedding": [-0.4, ..., 0.8]}
etc.
```
ここで、`id` フィールドの値は別のソースから実際のドキュメントを取得できるようにする識別子である必要があり、`embedding` の値はテキスト埋め込み API によって返されるベクトルです。

ドキュメントの `id` には、インメモリ ドキュメント ストアとして機能する `metadata` DataFrame の行インデックスをそのまま使用します。これにより、ベクトル検索によって返された `id` から要約、タイトル、URL を取得するのが非常に簡単になります。

```python
metadata.abstract[id]
metadata.title[id]
metadata.url[id]
```

次のセルでは `vectors` を反復処理し、`metadata` 内の要約のインデックスとテキスト埋め込み API によって返された埋め込みベクトルを含む上記のような JSON 行を `cord19_embeddings.json` に追加します。

In [ ]:
embeddings_file_path = "cord19_embeddings.json"

# 埋め込みファイルが既に存在する場合は削除
!test -f {embeddings_file_path} && rm {embeddings_file_path}

with open(embeddings_file_path, "a") as embeddings_file:
    for i, embedding in enumerate(vectors):
        json_line = json.dumps({"id": i, "embedding": embedding}) + "\n"
        embeddings_file.writelines(json_line)


埋め込みファイルに元の DataFrame と同じ行数（要約ごとに 1 行）があることを確認し、GCS バケットに保存します。

In [ ]:
!wc -l {embeddings_file_path}
print(len(metadata), "metadata dataframe")

In [ ]:
EMBEDDINGS_URI = f"gs://{BUCKET}"

!gcloud storage cp {embeddings_file_path} {EMBEDDINGS_URI}

## Vector Search エンジン インデックスの作成

それでは、[Vector Search](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/overview) のセットアップに取り掛かりましょう。この手順には次の 2 つのステップが必要です。

1. [インデックスの作成](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/create-manage-index)
1. [このインデックスのエンドポイントへのデプロイ](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/deploy-index-public)

インデックスの作成中に、埋め込みベクトルが Matching Engine にアップロードされ、指定されたベクトルの `approximate_neighbors_count` 個の最近傍を高速かつ近似的に検索できるツリー状のデータ構造（インデックス）が作成されます。インデックスは埋め込みベクトル間の距離の概念に依存しており、これを `distance_measure_type` で指定する必要があります。ここでは、基本的に埋め込みベクトル間の角度の尺度である `COSINE_DISTANCE`（コサイン距離）を選択します。他の可能な選択肢としては、ユークリッド距離の 2 乗（`SQUARED_L2_DISTANCE`）、[マンハッタン距離](https://en.wikipedia.org/wiki/Taxicab_geometry)（`L1_DISTANCE`）、またはドット積距離（`DOT_PRODUCT_DISTANCE`）があります。（使用している埋め込みが、一致するペア間のこれらの距離のいずれかを最小化するようにトレーニングされている場合は、その特定の距離を選択することでより良い結果が得られる可能性がありますが、それ以外の場合は `COSINE_DISTANCE` で十分です。）

次のセルでは、埋め込みファイルから Matching Engine インデックスを作成します。実行には約 1 時間かかります。

In [ ]:
DISPLAY_NAME = "cord19_embeddings"

matching_engine_index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name=DISPLAY_NAME,
    contents_delta_uri=EMBEDDINGS_URI,
    dimensions=len(vectors[0]),
    approximate_neighbors_count=150,
    distance_measure_type="COSINE_DISTANCE",
    leaf_node_embedding_count=500,
    leaf_nodes_to_search_percent=7,
    description=DISPLAY_NAME,
)

インデックスが作成されると、リソース名が関連付けられます。

In [ ]:
INDEX_RESOURCE_NAME = matching_engine_index.resource_name
print(INDEX_RESOURCE_NAME)

次に、このインデックス リソース名を使用してインデックスをインスタンス化できます。

In [ ]:
matching_engine_index = aiplatform.MatchingEngineIndex(
    index_name=INDEX_RESOURCE_NAME
)

インデックスが稼働したので、クエリを実行できるようにアクセス可能にする必要があります。最初のステップは、パブリック エンドポイントを作成することです（高速化のために、[VPC ネットワーク内にプライベート エンドポイント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/deploy-index-vpc) を作成することもできます）。

In [ ]:
matching_engine_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name=DISPLAY_NAME,
    description=DISPLAY_NAME,
    public_endpoint_enabled=True,
)

2 番目のステップは、作成したエンドポイントにインデックスをデプロイすることです。

In [ ]:
DEPLOYED_INDEX_ID = f"{DISPLAY_NAME}_deployed"

matching_engine = matching_engine_endpoint.deploy_index(
    index=matching_engine_index, deployed_index_id=DEPLOYED_INDEX_ID
)

matching_engine.deployed_indexes

## Vector Search のクエリ

これで Vector Search にクエリを発行する準備が整いました！

まず、ユーザー クエリからテキスト埋め込みを作成する必要があります。

In [ ]:
QUERY = "prophylactic measures"

embeddings = client.models.embed_content(model=EMBEDDING_MODEL, contents=QUERY)
text_embeddings = [vector.values for vector in embeddings.embeddings]

次に、デプロイされた Vector Search インデックスから `find_neighbors` メソッドを使用できます。このメソッドは、ユーザー クエリからの埋め込みベクトルを入力として受け取り、`NUM_NEIGHBORS` 個の最近傍の要約 ID を返します。

In [ ]:
# 返す近傍の数を定義
NUM_NEIGHBORS = 10

response = matching_engine.find_neighbors(
    deployed_index_id=DEPLOYED_INDEX_ID,
    queries=text_embeddings,
    num_neighbors=NUM_NEIGHBORS,
)

response


次のセルでは、最も関連性の高い `NUM_NEIGHBORS` 個の要約を、対応する論文のタイトルと URL も含む DataFrame にフォーマットします。

In [ ]:
matched_ids = [int(match.id) for match in response[0]]
matched_distances = [match.distance for match in response[0]]
matched_titles = [metadata.title[i] for i in matched_ids]
matched_abstracts = [metadata.abstract[i] for i in matched_ids]
matched_urls = [metadata.url[i] for i in matched_ids]

matches = pd.DataFrame(
    {
        "distance": matched_distances,
        "title": matched_titles,
        "abstract": matched_abstracts,
        "url": matched_urls,
    }
)
matches

利便性のためにシンプルなリストとしてフォーマットされた Vector Search のレスポンスを以下に示します。返された論文のリストの中に、クエリが英語であったにもかかわらず英語以外の言語の論文が含まれている場合があります。これは大規模言語モデルの多言語能力を示しており、マッチングが完全なキーワードの一致ではなく意味に基づいて行われていることを示しています。

In [ ]:
html = "<html><body><ol>"
for i in range(len(matches)):
    html += f"""            
    <li> 
        <article>
            <header>
                <a href="{matches.url[i]}"> <h2>{matches.title[i]}</h2></a>
            </header>
            <p>{matches.abstract[i]}</p>
        </article>
    </li>
    """
html += "</body></html>"
display.HTML(html)

## クリーンアップ

In [ ]:
matching_engine.delete(force=True)
matching_engine_index.delete()

Copyright 2023 Google Inc.
Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at
http://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.